# Part 2: Exploring Classification Algorithms

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [26]:
data = pd.read_csv('breast-cancer.csv')
data = data.drop(columns=['id'])

# Encode: M=1 (malignant), B=0 (benign)
data['diagnosis'] = data['diagnosis'].map({'M': 1, 'B': 0})

X = data.drop(columns=['diagnosis'])
y = data['diagnosis']

print(f"Shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts()}")
print(f"\nZeros per column:\n{(X == 0).sum().to_string()}")

Shape: (569, 30)
Class distribution:
diagnosis
0    357
1    212
Name: count, dtype: int64

Zeros per column:
radius_mean                 0
texture_mean                0
perimeter_mean              0
area_mean                   0
smoothness_mean             0
compactness_mean            0
concavity_mean             13
concave points_mean        13
symmetry_mean               0
fractal_dimension_mean      0
radius_se                   0
texture_se                  0
perimeter_se                0
area_se                     0
smoothness_se               0
compactness_se              0
concavity_se               13
concave points_se          13
symmetry_se                 0
fractal_dimension_se        0
radius_worst                0
texture_worst               0
perimeter_worst             0
area_worst                  0
smoothness_worst            0
compactness_worst           0
concavity_worst            13
concave points_worst       13
symmetry_worst              0
fractal_dimension_wo

In [27]:
# --- Replace 0's with NaN ---
X = data.drop(columns=['diagnosis'])
y = data['diagnosis']
X = X.replace(0, np.nan)

# print(f"NaN per column:\n{X.isna().sum().to_string()}")


In [28]:

# --- Compare imputation strategies ---
#Mean imputation
X_mean = X.copy()
mean_imputer = SimpleImputer(strategy='mean')
X_mean_imputed = mean_imputer.fit_transform(X_mean) 

#Median imputation
X_median = X.copy()
median_imputer = SimpleImputer(strategy='median')
X_median_imputed = median_imputer.fit_transform(X_median)

# Train/Predict Mean imputation
X_train_mean, X_test_mean, y_train_mean, y_test_mean = train_test_split(X_mean_imputed, y, test_size=0.3, random_state=33)
decision_tree_mean = DecisionTreeClassifier(max_depth=8, random_state=33)
decision_tree_mean.fit(X_train_mean, y_train_mean)
predictions_mean = decision_tree_mean.predict(X_test_mean)

# Evaluate Mean: accuracy, precision, recall, F1
accuracy_mean = accuracy_score(y_test_mean, predictions_mean)
precision_mean = precision_score(y_test_mean, predictions_mean)
recall_mean = recall_score(y_test_mean, predictions_mean)
f1_mean = f1_score(y_test_mean, predictions_mean)


# Train/Predict Mean imputation
X_train_median, X_test_median, y_train_median, y_test_median = train_test_split(X_median_imputed, y, test_size=0.3, random_state=33)
decision_tree_median = DecisionTreeClassifier(max_depth=8, random_state=33)
decision_tree_median.fit(X_train_median, y_train_median)
predictions_median = decision_tree_median.predict(X_test_median)

# Evaluate Median: accuracy, precision, recall, F1
accuracy_median = accuracy_score(y_test_median, predictions_median)
precision_median = precision_score(y_test_median, predictions_median)
recall_median = recall_score(y_test_median, predictions_median)
f1_median = f1_score(y_test_median, predictions_median)


# Print results
print("\n" + "=" * 40)
print("Mean Imputation:")
print(f"Accuracy: {accuracy_mean:.4f}")
print(f"Precision: {precision_mean:.4f}")
print(f"Recall: {recall_mean:.4f}")
print(f"F1 Score: {f1_mean:.4f}\n")
print("-" * 40 + "\n")
print("Median Imputation:")
print(f"Accuracy: {accuracy_median:.4f}")
print(f"Precision: {precision_median:.4f}")
print(f"Recall: {recall_median:.4f}")
print(f"F1 Score: {f1_median:.4f}")
print("\n" + "=" * 40)



# --- Step 2c: Pick the winner ---
# Choose whichever strategy gave better results
# Apply that imputation to the full dataset
# This becomes your clean X for all remaining steps
if accuracy_mean > accuracy_median:
    print("Mean imputation performed better. Using mean imputation for the rest of the analysis.")
    X_clean = X_mean_imputed
else:
    print("Median imputation performed better. Using median imputation for the rest of the analysis.")
    X_clean = X_median_imputed

X_clean = pd.DataFrame(X_clean, columns=X.columns)


Mean Imputation:
Accuracy: 0.9181
Precision: 0.8611
Recall: 0.9394
F1 Score: 0.8986

----------------------------------------

Median Imputation:
Accuracy: 0.9181
Precision: 0.8611
Recall: 0.9394
F1 Score: 0.8986

Median imputation performed better. Using median imputation for the rest of the analysis.


In [ ]:
# Reusable evaluation helper for all steps
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return {
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
    }

In [ ]:
# Step 3: Normalization comparison — KNN and Decision Tree across 3 scaling methods
X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.3, random_state=33)

results = []
for norm_name, scaler in [('None', None), ('MinMax', MinMaxScaler()), ('Standard', StandardScaler())]:
    Xtr = X_train.copy()
    Xte = X_test.copy()
    if scaler:
        Xtr = pd.DataFrame(scaler.fit_transform(Xtr), columns=X_clean.columns)
        Xte = pd.DataFrame(scaler.transform(Xte), columns=X_clean.columns)
    
    for model_name, model in [('KNN', KNeighborsClassifier(n_neighbors=9)), ('DT', DecisionTreeClassifier(max_depth=8, random_state=33))]:
        metrics = evaluate_model(model, Xtr, Xte, y_train, y_test)
        results.append({'Model': model_name, 'Normalization': norm_name, **metrics})

norm_results = pd.DataFrame(results)
print(norm_results.to_string(index=False))